In [5]:
import pandas as pd
import numpy as np
import json
import re
import ipaddress

In [6]:
with open("raw_data/track2_iam_audit_trail.json", "r", encoding="utf-8") as f:
    iam = json.load(f)

iam_df = pd.DataFrame(iam)

print("Shape:", iam_df.shape)
print("Columns:", iam_df.columns.tolist())

Shape: (20500, 15)
Columns: ['event_id', 'timestamp', 'user_id', 'username', 'department', 'event_type', 'auth_method', 'source_ip', 'hostname', 'device_id', 'session_id', 'mfa_passed', 'failure_reason', 'risk_score', 'geo_location']


In [7]:
iam_df.head()

,event_id,timestamp,user_id,username,department,event_type,auth_method,source_ip,hostname,device_id,session_id,mfa_passed,failure_reason,risk_score,geo_location
0,IAM00008974,05/09/2026 12:43,EMP-11889,atharv.shukla79,Compliance,LOGON_SUCCESS,otp,UNKNOWN,vdr-11889,DEV48365,SID120652,1,,78/100,PB
1,IAM00017969,09-Sep-2026 01:56:18,EMP-10918,isha.vyas39,Ops Team,MFA_FAILED,biometric,10-141.198,ws-10918,DEV79178,None,False,MFA failed,78,Maharashtra
2,IAM00015700,04/09/2026 04:04,EMP10629,ojasvi.majumdar45,Ops,FAILED_LOGIN,certificate,999.999.999.999,WS-10629,NA,None,True,NA,High,remote
3,IAM00016756,2026/09/09,emp_10271,widisha.nagy75,Legal Dept,PASSWORD_RESET,MFA,10.131.74,LPT-10271,DEV53236,None,False,None,78,Punjab
4,IAM00009357,2026-08-03T00:01:59,EMP10122,OSHA.CHAKRABORTY87,legal,MFA_FAILED,otp,172.16.14.,LPT_10122,DEV61717,None,no,NA,32,remote


In [8]:
print("=== IAM AUDIT TRAIL PROFILE ===")

print("Shape:", iam_df.shape)

print("\nData types:")
print(iam_df.dtypes)

print("\nMissing values:")
print(iam_df.isna().sum())

print("\nDuplicate rows:", iam_df.duplicated().sum())

print("\nDuplicate event IDs:", iam_df["event_id"].duplicated().sum())

=== IAM AUDIT TRAIL PROFILE ===
Shape: (20500, 15)

Data types:
event_id          object
timestamp         object
user_id           object
username          object
department        object
event_type        object
auth_method       object
source_ip         object
hostname          object
device_id         object
session_id        object
mfa_passed        object
failure_reason    object
risk_score        object
geo_location      object
dtype: object

Missing values:
event_id             0
timestamp            0
user_id              0
username             0
department        5758
event_type           0
auth_method          0
source_ip            0
hostname             0
device_id         1180
session_id        8677
mfa_passed           0
failure_reason    3621
risk_score        1062
geo_location      1768
dtype: int64

Duplicate rows: 500

Duplicate event IDs: 500


In [9]:
for col in [
    "event_type",
    "auth_method",
    "mfa_passed",
    "failure_reason"
]:
    print(f"\n=== {col.upper()} ===")
    print(iam_df[col].value_counts(dropna=False).head(30))


=== EVENT_TYPE ===
event_type
SSO_SUCCESS                     1769
success_login                   1715
AUTH_SUCCESS                    1698
LOGIN_SUCCESS                   1697
Successful Login                1696
Login Success                   1658
LOGON_SUCCESS                   1656
LOGIN_FAILED                     867
MFA_FAILED                       840
LOGON_FAILURE                    837
AUTH_FAILED                      809
FAILED_LOGIN                     785
Login Failed                     784
invalid_credentials              782
failed logon                     769
DEVICE_REGISTERED                320
ACCOUNT_UNLOCK                   318
SESSION_TERMINATED               310
PRIVILEGE_ESCALATION_REQUEST     303
MFA_ENROLLMENT                   301
ACCOUNT_LOCK                     299
PASSWORD_RESET                   287
Name: count, dtype: int64

=== AUTH_METHOD ===
auth_method
OTP            1761
MFA            1748
PASSWORD       1739
biometric      1734
password       1

In [10]:
print("=== USER ID SAMPLE ===")
print(iam_df["user_id"].head(30).tolist())

print("\n=== SOURCE IP SAMPLE ===")
print(iam_df["source_ip"].head(30).tolist())

print("\n=== HOSTNAME SAMPLE ===")
print(iam_df["hostname"].head(20).tolist())

print("\n=== DEVICE ID SAMPLE ===")
print(iam_df["device_id"].head(20).tolist())

=== USER ID SAMPLE ===
['EMP-11889', 'EMP-10918', 'EMP10629', 'emp_10271', 'EMP10122', 'EMP12961', 'EMP10289', 'EMP12196', 'EMP 12764', 'EMP11251', 'emp_11436', 'emp10912', 'EMP11632', 'emp12973', 'EMP12622', '12505', '12175', 'EMP12704', 'EMP10486', 'EMP10726', 'EMP12197', 'emp12144', 'EMP12056', 'EMP 10317', 'EMP 10795', 'EMP11969', 'emp11411', 'emp10845', 'EMP11001', 'EMP-10540']

=== SOURCE IP SAMPLE ===
['UNKNOWN', '10-141.198', '999.999.999.999', '10.131.74', '172.16.14.', '31.131.170.195', '20.56.54.159', '', 'UNKNOWN', '172.16.142.135.0', '192.168.42.', '24.235.2.', '36.141.191.113', '172.16.118.90', '107.242.96.202', '85.253.27.160', '999.999.999.999', '172.16.82.192', '192.168.75.75', '142.16.111.169', '172.16.58.169', '10.173.123', '', '192.168.70.191', '192.168.59.37', 'UNKNOWN', '10.0.204', '192.168.119.1', '172.16.85.218.0', '192.168.175.195']

=== HOSTNAME SAMPLE ===
['vdr-11889', 'ws-10918', 'WS-10629', 'LPT-10271', 'LPT_10122', 'LPT-12961', 'LPT-10289', 'LPT_12196', ''

In [11]:
print("=== TIMESTAMP SAMPLE ===")
print(iam_df["timestamp"].head(30).tolist())

=== TIMESTAMP SAMPLE ===
['05/09/2026 12:43', '09-Sep-2026 01:56:18', '04/09/2026 04:04', '2026/09/09', '2026-08-03T00:01:59', '2026-08-26 21:29:16', '10/08/2026', '1787085290', '2026-08-06T13:55:03', '22/08/2026', '13/08/2026 10:49', '2026-09-01 21:04:48', '08-15-2026 11:40:14 AM', '05/09/2026', '09-05-2026 07:18:17 AM', '01/09/2026 20:44', '04-Aug-2026 08:56:32', '2026-09-06 21:47:12', '2026-08-26 19:16:45', '', '08-13-2026 08:36:28 AM', '08-31-2026 06:20:13 PM', '', '', '06-Sep-2026 14:22:36', '07-Sep-2026 09:13:06', '2026/08/23', '2026-08-22 06:29:28', '2026-08-26 14:42:42', '2026-08-30 13:44:08']


In [12]:
# Remove exact duplicate rows
before = len(iam_df)
iam_df = iam_df.drop_duplicates().copy()
after = len(iam_df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicate rows removed:", before - after)

Rows before: 20500
Rows after: 20000
Duplicate rows removed: 500


In [13]:
# Clean IAM user IDs
iam_df["user_id_clean"] = (
    iam_df["user_id"]
    .astype("string")
    .str.upper()
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

print("Original user IDs:", iam_df["user_id"].nunique())
print("Clean user IDs:", iam_df["user_id_clean"].nunique())
print("Missing clean user IDs:", iam_df["user_id_clean"].isna().sum())

print("\nSample cleaned IDs:")
print(
    iam_df[["user_id", "user_id_clean"]]
    .head(20)
)

Original user IDs: 8785
Clean user IDs: 4158
Missing clean user IDs: 0

Sample cleaned IDs:
      user_id user_id_clean
0   EMP-11889      EMP11889
1   EMP-10918      EMP10918
2    EMP10629      EMP10629
3   emp_10271      EMP10271
4    EMP10122      EMP10122
5    EMP12961      EMP12961
6    EMP10289      EMP10289
7    EMP12196      EMP12196
8   EMP 12764      EMP12764
9    EMP11251      EMP11251
10  emp_11436      EMP11436
11   emp10912      EMP10912
12   EMP11632      EMP11632
13   emp12973      EMP12973
14   EMP12622      EMP12622
15      12505         12505
16      12175         12175
17   EMP12704      EMP12704
18   EMP10486      EMP10486
19   EMP10726      EMP10726


In [14]:
# Make numeric-only user IDs consistent with the EMP format

iam_df["user_id_clean"] = iam_df["user_id_clean"].apply(
    lambda x: f"EMP{x}" if pd.notna(x) and str(x).isdigit() else x
)

print("Sample cleaned IDs:")
print(
    iam_df[["user_id", "user_id_clean"]].head(10)
)

Sample cleaned IDs:
     user_id user_id_clean
0  EMP-11889      EMP11889
1  EMP-10918      EMP10918
2   EMP10629      EMP10629
3  emp_10271      EMP10271
4   EMP10122      EMP10122
5   EMP12961      EMP12961
6   EMP10289      EMP10289
7   EMP12196      EMP12196
8  EMP 12764      EMP12764
9   EMP11251      EMP11251


In [15]:
# Check whether one cleaned ID belongs to multiple original IDs
user_id_check = (
    iam_df[iam_df["user_id_clean"].notna()]
    .groupby("user_id_clean")["user_id"]
    .nunique()
)
print(
    "Clean IDs with multiple original ID formats:",
    (user_id_check > 1).sum()
)

Clean IDs with multiple original ID formats: 2690


In [16]:
iam_df["username_clean"] = (
    iam_df["username"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [17]:
# Treat blank usernames as missing values

iam_df["username_clean"] = iam_df["username_clean"].replace(
    r"^\s*$", pd.NA, regex=True
)

print("Missing usernames:", iam_df["username_clean"].isna().sum())
print("Unique cleaned usernames:", iam_df["username_clean"].nunique())

Missing usernames: 847
Unique cleaned usernames: 3868


In [18]:
username_check = (
    iam_df[iam_df["username_clean"].notna()]
    .groupby("username_clean")["user_id_clean"]
    .nunique()
)

print(
    "Real usernames linked to multiple user IDs:",
    (username_check > 1).sum()
)

Real usernames linked to multiple user IDs: 0


In [19]:
print("=== DEPARTMENT VALUES ===")

print(
    iam_df["department"]
    .value_counts(dropna=False)
)

=== DEPARTMENT VALUES ===
department
None                        5631
procurement team             352
Human Resources              345
Procurement                  344
Supply Chain                 342
Purchase                     336
Human Resource               334
hr dept                      333
HR                           329
IT                           326
People Team                  326
Information Technology       323
PURCH                        321
IT Dept                      318
IT Support                   318
customer care                309
CS                           308
Support                      304
information tech             297
Brand Team                   290
RD                           290
Call Center                  289
FINANCE                      288
Innovation                   282
Customer Support             281
Marketing                    280
marketing dept               280
OPERATIONS                   280
Ops                          280
Financ

In [20]:
department_map = {
    # Finance
    "Accounts": "Finance",
    "FINANCE": "Finance",
    "Fin": "Finance",
    "Finance": "Finance",
    "finance dept": "Finance",

    # Human Resources
    "HR": "Human Resources",
    "Human Resource": "Human Resources",
    "Human Resources": "Human Resources",
    "hr dept": "Human Resources",

    # IT
    "IT": "IT",
    "IT Dept": "IT",
    "IT Support": "IT",
    "Information Technology": "IT",
    "information tech": "IT",

    # Legal
    "LEGAL": "Legal",
    "Legal": "Legal",
    "Legal Dept": "Legal",
    "legal": "Legal",

    # Marketing
    "MKT": "Marketing",
    "Marketing": "Marketing",
    "Mktg": "Marketing",
    "marketing dept": "Marketing",

    # Operations
    "OPERATIONS": "Operations",
    "Operations": "Operations",
    "Ops": "Operations",
    "Ops Team": "Operations",
    "operations dept": "Operations",

    # Procurement
    "PURCH": "Procurement",
    "Procurement": "Procurement",
    "Purchase": "Procurement",
    "procurement team": "Procurement",

    # R&D
    "R&D": "R&D",
    "RD": "R&D",
    "Research and Development": "R&D",
    "RnD": "R&D",

    # Sales
    "SALES": "Sales",
    "Sales": "Sales",
    "Business Sales": "Sales",
    "sales dept": "Sales",
    "sales team": "Sales",

    # Customer Support
    "CS": "Customer Support",
    "Customer Support": "Customer Support",
    "Support": "Customer Support",
    "customer care": "Customer Support",
    "Call Center": "Customer Support",

    # Other standardized departments
    "Supply Chain": "Supply Chain",
    "People Team": "People",
    "Compliance": "Compliance",
    "Brand Team": "Brand",
    "Innovation": "Innovation"
}

iam_df["department_clean"] = iam_df["department"].map(department_map)

print("Missing original departments:",
      iam_df["department"].isna().sum())

print("Missing cleaned departments:",
      iam_df["department_clean"].isna().sum())

print("\nStandardized departments:")
print(iam_df["department_clean"].value_counts(dropna=False))

Missing original departments: 5631
Missing cleaned departments: 5631

Standardized departments:
department_clean
NaN                 5631
IT                  1582
Customer Support    1491
Operations          1362
Finance             1357
Procurement         1353
Human Resources     1341
Sales               1190
Marketing           1092
R&D                 1090
Legal                997
Supply Chain         342
People               326
Brand                290
Innovation           282
Compliance           274
Name: count, dtype: int64


In [21]:
#Event types
event_types = (
    iam_df["event_type"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Number of unique event types:", len(event_types))

for value in sorted(event_types):
    print(repr(value))

Number of unique event types: 22
'ACCOUNT_LOCK'
'ACCOUNT_UNLOCK'
'AUTH_FAILED'
'AUTH_SUCCESS'
'DEVICE_REGISTERED'
'FAILED_LOGIN'
'LOGIN_FAILED'
'LOGIN_SUCCESS'
'LOGON_FAILURE'
'LOGON_SUCCESS'
'Login Failed'
'Login Success'
'MFA_ENROLLMENT'
'MFA_FAILED'
'PASSWORD_RESET'
'PRIVILEGE_ESCALATION_REQUEST'
'SESSION_TERMINATED'
'SSO_SUCCESS'
'Successful Login'
'failed logon'
'invalid_credentials'
'success_login'


In [22]:
event_type_map = {
    # Successful authentication
    "AUTH_SUCCESS": "LOGIN_SUCCESS",
    "LOGIN_SUCCESS": "LOGIN_SUCCESS",
    "LOGON_SUCCESS": "LOGIN_SUCCESS",
    "SSO_SUCCESS": "LOGIN_SUCCESS",
    "Successful Login": "LOGIN_SUCCESS",
    "Login Success": "LOGIN_SUCCESS",
    "success_login": "LOGIN_SUCCESS",

    # Failed authentication
    "AUTH_FAILED": "LOGIN_FAILURE",
    "FAILED_LOGIN": "LOGIN_FAILURE",
    "LOGIN_FAILED": "LOGIN_FAILURE",
    "LOGON_FAILURE": "LOGIN_FAILURE",
    "Login Failed": "LOGIN_FAILURE",
    "failed logon": "LOGIN_FAILURE",
    "invalid_credentials": "LOGIN_FAILURE",

    # MFA
    "MFA_FAILED": "MFA_FAILURE",
    "MFA_ENROLLMENT": "MFA_ENROLLMENT",

    # Account management
    "ACCOUNT_LOCK": "ACCOUNT_LOCK",
    "ACCOUNT_UNLOCK": "ACCOUNT_UNLOCK",
    "PASSWORD_RESET": "PASSWORD_RESET",

    # Device / session
    "DEVICE_REGISTERED": "DEVICE_REGISTERED",
    "SESSION_TERMINATED": "SESSION_TERMINATED",

    # Privilege
    "PRIVILEGE_ESCALATION_REQUEST": "PRIVILEGE_ESCALATION_REQUEST"
}

iam_df["event_type_clean"] = iam_df["event_type"].map(event_type_map)

In [23]:
print("Missing cleaned event types:",
      iam_df["event_type_clean"].isna().sum())

print("\nStandardized event types:")
print(iam_df["event_type_clean"].value_counts(dropna=False))

Missing cleaned event types: 0

Standardized event types:
event_type_clean
LOGIN_SUCCESS                   11590
LOGIN_FAILURE                    5509
MFA_FAILURE                       815
ACCOUNT_UNLOCK                    312
DEVICE_REGISTERED                 311
SESSION_TERMINATED                303
MFA_ENROLLMENT                    295
PRIVILEGE_ESCALATION_REQUEST      293
ACCOUNT_LOCK                      289
PASSWORD_RESET                    283
Name: count, dtype: int64


In [24]:
auth_methods = (
    iam_df["auth_method"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Number of unique auth methods:", len(auth_methods))

for value in sorted(auth_methods):
    print(repr(value))

Number of unique auth methods: 12
'BIOMETRIC'
'Certificate'
'MFA'
'OTP'
'PASSWORD'
'SSO'
'biometric'
'certificate'
'mfa'
'otp'
'password'
'sso'


In [25]:
auth_method_map = {
    "BIOMETRIC": "BIOMETRIC",
    "biometric": "BIOMETRIC",

    "Certificate": "CERTIFICATE",
    "certificate": "CERTIFICATE",

    "MFA": "MFA",
    "mfa": "MFA",

    "OTP": "OTP",
    "otp": "OTP",

    "PASSWORD": "PASSWORD",
    "password": "PASSWORD",

    "SSO": "SSO",
    "sso": "SSO"
}

iam_df["auth_method_clean"] = iam_df["auth_method"].map(auth_method_map)

In [26]:
print("Missing cleaned auth methods:",
      iam_df["auth_method_clean"].isna().sum())

print("\nStandardized auth methods:")
print(iam_df["auth_method_clean"].value_counts(dropna=False))

unmapped_auth = iam_df.loc[
    iam_df["auth_method"].notna() &
    iam_df["auth_method_clean"].isna(),
    "auth_method"
].unique()

print("\nUnmapped auth methods:", len(unmapped_auth))
print(unmapped_auth)

Missing cleaned auth methods: 0

Standardized auth methods:
auth_method_clean
OTP            3376
PASSWORD       3373
MFA            3369
SSO            3307
CERTIFICATE    3290
BIOMETRIC      3285
Name: count, dtype: int64

Unmapped auth methods: 0
[]


In [27]:
def clean_iam_timestamp(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Empty / missing values
    if value == "":
        return pd.NaT

    # Unix timestamp in seconds
    if re.fullmatch(r"\d{10}", value):
        try:
            return pd.to_datetime(int(value), unit="s")
        except:
            return pd.NaT

    # ISO format: 2026-08-03T00:01:59
    if re.match(r"^\d{4}-\d{2}-\d{2}", value):
        try:
            return pd.to_datetime(value, dayfirst=False)
        except:
            return pd.NaT

    # YYYY/MM/DD format
    if re.match(r"^\d{4}/\d{2}/\d{2}", value):
        try:
            return pd.to_datetime(value, dayfirst=False)
        except:
            return pd.NaT

    # DD/MM/YYYY format
    if re.match(r"^\d{2}/\d{2}/\d{4}", value):
        try:
            return pd.to_datetime(value, dayfirst=True)
        except:
            return pd.NaT

    # DD-Mon-YYYY format
    if re.match(r"^\d{1,2}-[A-Za-z]{3}-\d{4}", value):
        try:
            return pd.to_datetime(value, dayfirst=True)
        except:
            return pd.NaT

    # MM-DD-YYYY with AM/PM
    if re.match(r"^\d{2}-\d{2}-\d{4}.*(?:AM|PM)$", value, re.IGNORECASE):
        try:
            return pd.to_datetime(value, dayfirst=False)
        except:
            return pd.NaT

    # Fallback
    try:
        return pd.to_datetime(value, dayfirst=False)
    except:
        return pd.NaT


iam_df["timestamp_clean"] = iam_df["timestamp"].apply(
    clean_iam_timestamp
)

In [28]:
print("Earliest timestamp:",
      iam_df["timestamp_clean"].min())

print("Latest timestamp:",
      iam_df["timestamp_clean"].max())

Earliest timestamp: 2026-08-01 00:00:00
Latest timestamp: 2026-09-09 23:55:40


In [29]:
# Timestamp quality validation

missing_timestamp_count = iam_df["timestamp_clean"].isna().sum()

invalid_range = iam_df[
    iam_df["timestamp_clean"].notna() &
    (
        (iam_df["timestamp_clean"] < "2026-08-01") |
        (iam_df["timestamp_clean"] > "2026-09-09 23:59:59")
    )
].shape[0]

print("Missing timestamps:", missing_timestamp_count)
print("Timestamps outside expected range:", invalid_range)
print("Earliest valid timestamp:", iam_df["timestamp_clean"].min())
print("Latest valid timestamp:", iam_df["timestamp_clean"].max())

Missing timestamps: 2416
Timestamps outside expected range: 0
Earliest valid timestamp: 2026-08-01 00:00:00
Latest valid timestamp: 2026-09-09 23:55:40


In [30]:
ip_text = iam_df["source_ip"].astype("string").str.strip()

print("Blank IPs:", (ip_text == "").sum())
print("UNKNOWN IPs:", (ip_text.str.upper() == "UNKNOWN").sum())
print("Missing/None IPs:", iam_df["source_ip"].isna().sum())

Blank IPs: 1169
UNKNOWN IPs: 1178
Missing/None IPs: 0


In [31]:
def classify_ip(value):
    if pd.isna(value):
        return "MISSING"

    value = str(value).strip()

    if value == "":
        return "MISSING"

    if value.upper() in ["UNKNOWN", "N/A", "NA", "NONE", "NULL"]:
        return "UNKNOWN"

    try:
        ip = ipaddress.ip_address(value)

        if ip.version == 4:
            if ip.is_private:
                return "VALID_PRIVATE"
            else:
                return "VALID_PUBLIC"

        return "VALID_OTHER"

    except ValueError:
        return "MALFORMED"


iam_df["ip_status"] = iam_df["source_ip"].apply(classify_ip)

In [32]:
print("=== IP QUALITY ===")
print(iam_df["ip_status"].value_counts(dropna=False))

=== IP QUALITY ===
ip_status
MALFORMED        7770
VALID_PRIVATE    6171
VALID_PUBLIC     3712
UNKNOWN          1178
MISSING          1169
Name: count, dtype: int64


In [33]:
ip_event_summary = pd.crosstab(
    iam_df["event_type_clean"],
    iam_df["ip_status"]
)

print(ip_event_summary)

ip_status                     MALFORMED  MISSING  UNKNOWN  VALID_PRIVATE  \
event_type_clean                                                           
ACCOUNT_LOCK                        104       15       12             99   
ACCOUNT_UNLOCK                      128       21       18             93   
DEVICE_REGISTERED                   131       12       19             95   
LOGIN_FAILURE                      2057      322      344           1721   
LOGIN_SUCCESS                      4595      674      657           3565   
MFA_ENROLLMENT                      104       18       18             84   
MFA_FAILURE                         321       48       57            245   
PASSWORD_RESET                      102       20       18             93   
PRIVILEGE_ESCALATION_REQUEST        111       14       19             86   
SESSION_TERMINATED                  117       25       16             90   

ip_status                     VALID_PUBLIC  
event_type_clean                          

In [34]:
ip_event_pct = pd.crosstab(
    iam_df["event_type_clean"],
    iam_df["ip_status"],
    normalize="index"
) * 100

print(ip_event_pct.round(1))

ip_status                     MALFORMED  MISSING  UNKNOWN  VALID_PRIVATE  \
event_type_clean                                                           
ACCOUNT_LOCK                       36.0      5.2      4.2           34.3   
ACCOUNT_UNLOCK                     41.0      6.7      5.8           29.8   
DEVICE_REGISTERED                  42.1      3.9      6.1           30.5   
LOGIN_FAILURE                      37.3      5.8      6.2           31.2   
LOGIN_SUCCESS                      39.6      5.8      5.7           30.8   
MFA_ENROLLMENT                     35.3      6.1      6.1           28.5   
MFA_FAILURE                        39.4      5.9      7.0           30.1   
PASSWORD_RESET                     36.0      7.1      6.4           32.9   
PRIVILEGE_ESCALATION_REQUEST       37.9      4.8      6.5           29.4   
SESSION_TERMINATED                 38.6      8.3      5.3           29.7   

ip_status                     VALID_PUBLIC  
event_type_clean                          

In [35]:
def clean_iam_hostname(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    if value == "":
        return pd.NA

    value = value.upper()

    # Remove corporate domain suffix
    value = re.sub(r"\.CORP\.LOCAL$", "", value)

    # Standardize underscore to hyphen
    value = value.replace("_", "-")

    return value


iam_df["hostname_clean"] = iam_df["hostname"].apply(
    clean_iam_hostname
)

In [36]:
hostname_check = (
    iam_df[iam_df["hostname_clean"].notna()]
    .groupby("hostname_clean")["user_id_clean"]
    .nunique()
)

print(
    "Hostnames linked to multiple users:",
    (hostname_check > 1).sum()
)

Hostnames linked to multiple users: 0


In [37]:
print("Number of unique device IDs:",
      iam_df["device_id"].nunique(dropna=False))

print("\nMost common device ID values:")
print(
    iam_df["device_id"]
    .value_counts(dropna=False)
    .head(30)
)

Number of unique device IDs: 2915

Most common device ID values:
device_id
            1206
NA          1168
None        1154
DEV98652      69
DEV82559      68
DEV97274      68
DEV24748      66
DEV21792      66
DEV66547      64
DEV67850      64
DEV81188      64
DEV59792      63
DEV96017      62
DEV46068      62
DEV59411      62
DEV75210      61
DEV81061      61
DEV97562      60
DEV26119      60
DEV96424      60
DEV60738      58
DEV96661      58
DEV47712      58
DEV46747      57
DEV78025      57
DEV90702      57
DEV94267      57
DEV18559      57
DEV68365      57
DEV94012      56
Name: count, dtype: int64


In [38]:
def clean_iam_device_id(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip()

    # Treat placeholders/blanks as missing
    if value.upper() in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA

    # Standardize case and remove spaces/hyphens
    value = value.upper()
    value = re.sub(r"[\s-]+", "", value)

    return value


iam_df["device_id_clean"] = iam_df["device_id"].apply(
    clean_iam_device_id
)

In [39]:
device_user_counts = (
    iam_df[iam_df["device_id_clean"].notna()]
    .groupby("device_id_clean")["user_id_clean"]
    .nunique()
)

print(
    "Device IDs linked to multiple users:",
    (device_user_counts > 1).sum()
)

print(
    "IAM records affected:",
    iam_df["device_id_clean"].isin(
        device_user_counts[device_user_counts > 1].index
    ).sum()
)

Device IDs linked to multiple users: 50
IAM records affected: 658


In [40]:
def clean_iam_session_id(value):
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip()
    
    if value.upper() in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA
    
    return value.upper()

iam_df["session_id_clean"] = iam_df["session_id"].apply(
    clean_iam_session_id
)

print("Missing cleaned session IDs:",
      iam_df["session_id_clean"].isna().sum())

print("Unique cleaned session IDs:",
      iam_df["session_id_clean"].nunique())

print("\nSample:")
print(iam_df[["session_id", "session_id_clean"]].head(20))

Missing cleaned session IDs: 8476
Unique cleaned session IDs: 11452

Sample:
   session_id session_id_clean
0   SID120652        SID120652
1        None             <NA>
2        None             <NA>
3        None             <NA>
4        None             <NA>
5        None             <NA>
6   SID754956        SID754956
7        None             <NA>
8        None             <NA>
9   SID567692        SID567692
10  SID659017        SID659017
11       None             <NA>
12       None             <NA>
13  SID662830        SID662830
14       None             <NA>
15       None             <NA>
16  SID494071        SID494071
17       None             <NA>
18  SID505840        SID505840
19       None             <NA>


In [41]:
def clean_mfa_passed(value):
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip().upper()
    
    if value in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA
    
    if value in ["TRUE", "YES", "Y", "1", "PASSED", "PASS"]:
        return True
    
    if value in ["FALSE", "NO", "N", "0", "FAILED", "FAIL"]:
        return False
    
    return pd.NA

iam_df["mfa_passed_clean"] = iam_df["mfa_passed"].apply(
    clean_mfa_passed
)

print("MFA values:")
print(iam_df["mfa_passed_clean"].value_counts(dropna=False))

print("\nOriginal vs cleaned:")
print(
    iam_df[["mfa_passed", "mfa_passed_clean"]]
    .drop_duplicates()
    .sort_values("mfa_passed_clean", na_position="last")
)

MFA values:
mfa_passed_clean
True     12256
False     7744
Name: count, dtype: int64

Original vs cleaned:
   mfa_passed  mfa_passed_clean
1       False             False
3       False             False
4          no             False
11          N             False
0           1              True
6           Y              True
8         yes              True
18       True              True


In [42]:
def clean_failure_reason(value):
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip()
    
    if value.upper() in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA
    
    return value.upper()

iam_df["failure_reason_clean"] = iam_df["failure_reason"].apply(
    clean_failure_reason
)

print("Missing failure reasons:",
      iam_df["failure_reason_clean"].isna().sum())

print("\nUnique failure reasons:",
      iam_df["failure_reason_clean"].nunique())

print("\nFailure reason counts:")
print(iam_df["failure_reason_clean"].value_counts(dropna=False))

Missing failure reasons: 15146

Unique failure reasons: 10

Failure reason counts:
failure_reason_clean
<NA>                   15146
INVALID CREDENTIALS      532
TIMEOUT                  522
UNKNOWN USER             499
BAD TOKEN                487
ACCOUNT LOCKED           483
WRONG_PASSWORD           476
OTP EXPIRED              465
MFA FAILED               465
EXPIRED PASSWORD         463
WRONG PASSWORD           462
Name: count, dtype: int64


In [43]:
print("Unique raw risk scores:")
print(iam_df["risk_score"].value_counts(dropna=False))

print("\nData type:")
print(iam_df["risk_score"].dtype)

Unique raw risk scores:
risk_score
None      1037
High       291
HIGH       271
medium     268
Medium     266
          ... 
216          5
198          5
197          4
200          4
122          3
Name: count, Length: 469, dtype: int64

Data type:
object


In [44]:
def clean_risk_score(value):
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip().upper()
    
    if value in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA
    
    # Direct risk labels
    if value == "LOW":
        return "LOW"
    if value == "MEDIUM":
        return "MEDIUM"
    if value == "HIGH":
        return "HIGH"
    
    # Numeric score
    try:
        score = float(value)
        
        # Accept only sensible risk-score values
        if 0 <= score <= 100:
            return score
        
    except ValueError:
        pass
    
    return pd.NA


iam_df["risk_score_clean"] = iam_df["risk_score"].apply(
    clean_risk_score
)

print("Cleaned risk score values:")
print(iam_df["risk_score_clean"].value_counts(dropna=False).head(20))

print("\nMissing cleaned risk scores:",
      iam_df["risk_score_clean"].isna().sum())

Cleaned risk score values:
risk_score_clean
<NA>      5266
HIGH       562
MEDIUM     534
LOW        257
15.0       165
78.0       157
84.0       155
42.0       155
65.0       155
68.0       154
72.0       153
64.0       152
98.0       149
55.0       147
47.0       147
2.0        147
97.0       146
81.0       146
80.0       145
86.0       145
Name: count, dtype: int64

Missing cleaned risk scores: 5266


In [45]:
# Numeric risk score
iam_df["risk_score_numeric"] = pd.to_numeric(
    iam_df["risk_score_clean"],
    errors="coerce"
)

# Risk level from the original categorical labels
iam_df["risk_level_clean"] = (
    iam_df["risk_score_clean"]
    .astype("string")
    .str.upper()
)

iam_df.loc[
    ~iam_df["risk_level_clean"].isin(["LOW", "MEDIUM", "HIGH"]),
    "risk_level_clean"
] = pd.NA

print("Numeric risk scores:")
print(iam_df["risk_score_numeric"].describe())

print("\nRisk levels:")
print(iam_df["risk_level_clean"].value_counts(dropna=False))

Numeric risk scores:
count    13381.000000
mean        50.242508
std         29.198399
min          0.000000
25%         25.000000
50%         51.000000
75%         76.000000
max        100.000000
Name: risk_score_numeric, dtype: float64

Risk levels:
risk_level_clean
<NA>      18647
HIGH        562
MEDIUM      534
LOW         257
Name: count, dtype: Int64


In [46]:
print("Unique geo locations:",
      iam_df["geo_location"].nunique(dropna=True))

print("\nRaw geo location values:")
print(iam_df["geo_location"].value_counts(dropna=False))

Unique geo locations: 11

Raw geo location values:
geo_location
remote         1736
None           1723
Punjab         1707
IN             1702
India          1671
Delhi          1668
DL             1650
               1649
UNKNOWN        1627
PB             1626
ind            1625
Maharashtra    1616
Name: count, dtype: int64


In [47]:
def clean_geo_location(value):
    if pd.isna(value):
        return pd.NA
    
    value = str(value).strip().upper()
    
    if value in ["", "NONE", "N/A", "NA"]:
        return pd.NA
    
    if value == "UNKNOWN":
        return "UNKNOWN"
    
    if value == "REMOTE":
        return "REMOTE"
    
    if value in ["IN", "INDIA", "IND"]:
        return "INDIA"
    
    if value in ["PB", "PUNJAB"]:
        return "PUNJAB"
    
    if value in ["DL", "DELHI"]:
        return "DELHI"
    
    if value == "MAHARASHTRA":
        return "MAHARASHTRA"
    
    return value


iam_df["geo_location_clean"] = iam_df["geo_location"].apply(
    clean_geo_location
)

print("Cleaned geo locations:")
print(iam_df["geo_location_clean"].value_counts(dropna=False))

Cleaned geo locations:
geo_location_clean
INDIA          4998
<NA>           3372
PUNJAB         3333
DELHI          3318
REMOTE         1736
UNKNOWN        1627
MAHARASHTRA    1616
Name: count, dtype: int64


In [48]:
# ==========================================
# FINAL IAM STANDARDIZATION
# ==========================================

# 1. Timestamp
iam_df["timestamp"] = pd.to_datetime(
    iam_df["timestamp_clean"],
    errors="coerce"
)

# 2. User ID
iam_df["user_id"] = iam_df["user_id_clean"]

# 3. Username
iam_df["username"] = iam_df["username_clean"]

# 4. Department
iam_df["department"] = iam_df["department_clean"]

# 5. Event type
iam_df["event_type"] = iam_df["event_type_clean"]

# 6. Authentication method
iam_df["auth_method"] = iam_df["auth_method_clean"]

# 7. Hostname
iam_df["hostname"] = iam_df["hostname_clean"]

# 8. Device ID
iam_df["device_id"] = iam_df["device_id_clean"]

# 9. Session ID
iam_df["session_id"] = iam_df["session_id_clean"]

# 10. MFA
iam_df["mfa_passed"] = iam_df["mfa_passed_clean"]

# 11. Failure reason
failure_map = {
    "INVALID CREDENTIALS": "INVALID_CREDENTIALS",
    "TIMEOUT": "TIMEOUT",
    "UNKNOWN USER": "UNKNOWN_USER",
    "BAD TOKEN": "BAD_TOKEN",
    "ACCOUNT LOCKED": "ACCOUNT_LOCKED",
    "WRONG_PASSWORD": "WRONG_PASSWORD",
    "WRONG PASSWORD": "WRONG_PASSWORD",
    "OTP EXPIRED": "OTP_EXPIRED",
    "MFA FAILED": "MFA_FAILED",
    "EXPIRED PASSWORD": "EXPIRED_PASSWORD"
}

iam_df["failure_reason"] = (
    iam_df["failure_reason_clean"]
    .map(failure_map)
)

# 12. Risk score
def clean_final_risk(value):
    if pd.isna(value):
        return pd.NA

    value = str(value).strip().upper()

    if value in ["", "NA", "N/A", "NONE", "NULL", "UNKNOWN"]:
        return pd.NA

    # Values such as 78/100
    match = re.fullmatch(
        r"(\d+(?:\.\d+)?)\s*/\s*100",
        value
    )

    if match:
        score = float(match.group(1))
        if 0 <= score <= 100:
            return score

    # Plain numeric score
    try:
        score = float(value)

        if 0 <= score <= 100:
            return score

    except ValueError:
        pass

    return pd.NA


iam_df["risk_score_numeric"] = iam_df["risk_score"].apply(
    clean_final_risk
)

# Keep explicit HIGH / MEDIUM / LOW labels separately
iam_df["risk_level"] = (
    iam_df["risk_score"]
    .astype("string")
    .str.strip()
    .str.upper()
)

iam_df.loc[
    ~iam_df["risk_level"].isin(["LOW", "MEDIUM", "HIGH"]),
    "risk_level"
] = pd.NA

# 13. Geo location
iam_df["geo_location"] = iam_df["geo_location_clean"]

# 14. Source IP stays raw because malformed IPs
# are important cybersecurity signals.
iam_df["source_ip"] = (
    iam_df["source_ip"]
    .astype("string")
    .str.strip()
)

# ==========================================
# KEEP ONLY FINAL ANALYSIS COLUMNS
# ==========================================

final_cols = [
    "event_id",
    "timestamp",
    "user_id",
    "username",
    "department",
    "event_type",
    "auth_method",
    "source_ip",
    "ip_status",
    "hostname",
    "device_id",
    "session_id",
    "mfa_passed",
    "failure_reason",
    "risk_score_numeric",
    "risk_level",
    "geo_location"
]

iam_final = iam_df[final_cols].copy()

print("Final IAM shape:", iam_final.shape)
print("\nFinal columns:")
print(iam_final.columns.tolist())

Final IAM shape: (20000, 17)

Final columns:
['event_id', 'timestamp', 'user_id', 'username', 'department', 'event_type', 'auth_method', 'source_ip', 'ip_status', 'hostname', 'device_id', 'session_id', 'mfa_passed', 'failure_reason', 'risk_score_numeric', 'risk_level', 'geo_location']


In [49]:
print("===== FINAL IAM QUALITY CHECK =====")

print("\nShape:")
print(iam_final.shape)

print("\nDuplicate rows:")
print(iam_final.duplicated().sum())

print("\nDuplicate event IDs:")
print(iam_final["event_id"].duplicated().sum())

print("\nData types:")
print(iam_final.dtypes)

print("\nMissing values:")
print(iam_final.isna().sum())

print("\nEvent types:")
print(iam_final["event_type"].value_counts(dropna=False))

print("\nAuth methods:")
print(iam_final["auth_method"].value_counts(dropna=False))

print("\nDepartments:")
print(iam_final["department"].value_counts(dropna=False))

print("\nMFA:")
print(iam_final["mfa_passed"].value_counts(dropna=False))

print("\nFailure reasons:")
print(iam_final["failure_reason"].value_counts(dropna=False))

print("\nRisk scores:")
print(iam_final["risk_score_numeric"].describe())

print("\nRisk levels:")
print(iam_final["risk_level"].value_counts(dropna=False))

print("\nGeo locations:")
print(iam_final["geo_location"].value_counts(dropna=False))

===== FINAL IAM QUALITY CHECK =====

Shape:
(20000, 17)

Duplicate rows:
0

Duplicate event IDs:
0

Data types:
event_id                      object
timestamp             datetime64[ns]
user_id                       object
username              string[python]
department                    object
event_type                    object
auth_method                   object
source_ip             string[python]
ip_status                     object
hostname                      object
device_id                     object
session_id                    object
mfa_passed                      bool
failure_reason                object
risk_score_numeric            object
risk_level            string[python]
geo_location                  object
dtype: object

Missing values:
event_id                  0
timestamp              2416
user_id                   0
username                847
department             5631
event_type                0
auth_method               0
source_ip                 0
ip_s

In [50]:
print("===== IAM FINAL AUDIT =====")

print("Rows:", len(iam_df))
print("Columns:", len(iam_df.columns))

print("\nDuplicate rows:", iam_df.duplicated().sum())
print("Duplicate event IDs:", iam_df["event_id"].duplicated().sum())

print("\nMissing values in cleaned columns:")
cleaned_cols = [
    "user_id_clean",
    "username_clean",
    "department_clean",
    "event_type_clean",
    "auth_method_clean",
    "timestamp_clean",
    "device_id_clean",
    "session_id_clean",
    "mfa_passed_clean",
    "failure_reason_clean",
    "risk_score_numeric",
    "risk_level_clean",
    "geo_location_clean"
]

print(iam_df[cleaned_cols].isna().sum())

print("\nRisk score range:")
print(
    iam_df["risk_score_numeric"].min(),
    "to",
    iam_df["risk_score_numeric"].max()
)

print("\nTimestamp range:")
print(iam_df["timestamp_clean"].min())
print(iam_df["timestamp_clean"].max())

===== IAM FINAL AUDIT =====
Rows: 20000
Columns: 32

Duplicate rows: 0
Duplicate event IDs: 0

Missing values in cleaned columns:
user_id_clean               0
username_clean            847
department_clean         5631
event_type_clean            0
auth_method_clean           0
timestamp_clean          2416
device_id_clean          3528
session_id_clean         8476
mfa_passed_clean            0
failure_reason_clean    15146
risk_score_numeric       4997
risk_level_clean        18647
geo_location_clean       3372
dtype: int64

Risk score range:
0.0 to 100.0

Timestamp range:
2026-08-01 00:00:00
2026-09-09 23:55:40


In [51]:
iam_final["risk_score_numeric"] = pd.to_numeric(
    iam_final["risk_score_numeric"],
    errors="coerce"
)

print(iam_final["risk_score_numeric"].dtype)
print(iam_final["risk_score_numeric"].describe())

float64
count    15003.000000
mean        50.222356
std         29.116161
min          0.000000
25%         25.000000
50%         50.000000
75%         76.000000
max        100.000000
Name: risk_score_numeric, dtype: float64


In [52]:
# ==========================================
# FILL MISSING IAM DATA FROM IDENTITY MASTER
# ==========================================

identity_enriched = pd.read_csv(
    "cleaned_data/identity_asset_master_clean.csv"
)

print("Identity shape:", identity_enriched.shape)

Identity shape: (3000, 14)


In [53]:
identity_reference = identity_enriched[
    ["user_id", "username", "department", "hostname", "device_id"]
].copy()

In [54]:
iam_filled = iam_final.merge(
    identity_reference,
    on="user_id",
    how="left",
    suffixes=("", "_identity")
)

print("IAM shape after merge:", iam_filled.shape)

IAM shape after merge: (20000, 21)


In [55]:
for col in ["username", "department", "hostname", "device_id"]:
    iam_filled[col] = iam_filled[col].fillna(
        iam_filled[col + "_identity"]
    )

In [56]:
iam_filled = iam_filled.drop(
    columns=[
        "username_identity",
        "department_identity",
        "hostname_identity",
        "device_id_identity"
    ]
)

In [57]:
print("Missing values after Identity enrichment:")

print(
    iam_filled[
        ["username", "department", "hostname", "device_id"]
    ].isna().sum()
)

Missing values after Identity enrichment:
username       12
department      0
hostname       37
device_id     275
dtype: int64


In [58]:
print("\nRows:", len(iam_filled))
print("Duplicate event IDs:",
      iam_filled["event_id"].duplicated().sum())


Rows: 20000
Duplicate event IDs: 0


In [62]:
# Save final enriched IAM dataset

iam_filled.to_csv(
    "cleaned_data/iam_audit_clean.csv",
    index=False
)

print("Saved successfully!")
print("Shape:", iam_filled.shape)

Saved successfully!
Shape: (20000, 17)
